# 03｜基準評估：SQuAD（300 題）＋ HotpotQA（300 題）

這本是 Stage 3 的**比較基準**。之後換成混合檢索時，只改第 0 節的 `RETRIEVER`，其他全部不動再跑一次，兩次結果放在一起比。

| 資料集 | 知識庫 | 題目 | 特性 |
|---|---|---|---|
| SQuAD | `squad_v11_dev500`（534 段） | `squad_dev300.jsonl`（前 50 題就是之前的 50 題） | 一題一個正確段落，題目用字和段落很像 |
| HotpotQA | `hotpot_dev400`（約 4,000 段） | `hotpot_dev.jsonl` | **一題要兩個段落**才答得出來；干擾段落是刻意挑過、看起來很像的 |

**兩層評估：**

1. **檢索**：全部 300 題，不呼叫 LLM，結果每次都一樣。這是判斷檢索方法好壞最可靠的數字。
2. **完整問答**：每個資料集前 `N_E2E` 題，跑 RAG 和 oracle（直接給正確段落），再加上 50 題無答案題（Collection B 的題目拿去問 A）。LLM 每次結果會有些許不同。

**HotpotQA 的「找到」：** 兩個正確段落**都**進了 Top-k 才算找到（`hit@k`）；只找到一個的另外記為 `any@k`。

**執行時間：** 檢索幾分鐘（HotpotQA 第一次要建約 4,000 段的索引）。完整問答 `N_E2E = 150` 時約 300×2＋50 次 LLM 呼叫，4B 約 30 分鐘；`N_E2E = 0` 只跑 50 題無答案題，約 2 分鐘。

**`SPLIT`：** 開發時用 `"dev"`；所有設定定案後，dense 和 hybrid 各用 `"holdout"` 跑一次（SQuAD 100、HotpotQA 100、無答案 50），確認結論在沒看過的題目上也成立。

## 0. 設定

In [12]:
import json
import sys
import time
from datetime import datetime
from pathlib import Path

import numpy as np
import pandas as pd

ROOT = Path.cwd()
if not (ROOT / "collections").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
from src import rag_core as rc

SPLIT = "dev"                  # "dev"：開發用。"holdout"：設定全部定案後，最後驗收才跑

RETRIEVER = "dense"            # "dense" | "bm25" | "hybrid"（dense + BM25，用 RRF 融合）
RRF_K = 60                     # 只有 hybrid 用到
EMBED_MODEL = "BAAI/bge-small-en-v1.5"
GEN_MODEL, GEN_THINK = "qwen3:4b-instruct", None
OLLAMA_URL = "http://localhost:11434"
NUM_CTX = 4096
CONTEXT_CHAR_BUDGET = 8000
TOP_K = 3                      # 和之前相同的設定
RETRIEVAL_KS = (1, 2, 3, 5, 10)
N_E2E = 150                    # 完整問答每個資料集用幾題；先用 5 試跑。0 = 只跑無答案題
RUN_E2E = True                 # False：只做第 3 節的檢索評估（幾分鐘，不呼叫 LLM）
RUN_NO_RAG = False             # True：另外跑 no_rag，檢查模型是不是本來就背過這些題目（預訓練污染）
CORRECT_F1 = 0.5

LOAD_RUN = None                # 填既有 run_id 就只讀紀錄畫表，不呼叫 LLM
BASELINE_RUN = None            # 填基準 run_id（例如 dense 那次），第 7 節會並排比較

SPLIT="holdout"
RETRIEVER="hybrid"
N_E2E=100
BASELINE_RUN="20261026_185546_bench_dense_holdout"

pd.set_option("display.max_colwidth", 90)
print("ROOT =", ROOT)

ROOT = <專案根目錄>


## 1. 載入資料

In [13]:
# 題目檔依 SPLIT 決定。放在這裡（而不是第 0 節），所以在第 0 節任何位置改 SPLIT 都有效。
QUESTION_FILES = {
    "dev":     {"squad": "squad_dev300.jsonl",     "hotpot": "hotpot_dev.jsonl",
                "noanswer": "squad_b_noanswer_dev50.jsonl"},
    "holdout": {"squad": "squad_holdout100.jsonl", "hotpot": "hotpot_holdout.jsonl",
                "noanswer": "squad_b_noanswer_holdout50.jsonl"},
}[SPLIT]
DATASETS = {
    "squad":  {"collection": "squad_v11_dev500", "questions": QUESTION_FILES["squad"]},
    "hotpot": {"collection": "hotpot_dev400",    "questions": QUESTION_FILES["hotpot"]},
}
NOANSWER = {"questions": QUESTION_FILES["noanswer"], "ask_collection": "squad_v11_dev500"}   # B 的題目問 A
assert all(SPLIT in f for f in QUESTION_FILES.values()), f"題目檔和 SPLIT={SPLIT} 不符：{QUESTION_FILES}"
print(f"SPLIT = {SPLIT}　RETRIEVER = {RETRIEVER}　N_E2E = {N_E2E}　題目檔：{list(QUESTION_FILES.values())}")

data = {}
for name, cfg in DATASETS.items():
    folder = ROOT / "collections" / cfg["collection"]
    qfile = ROOT / "eval" / cfg["questions"]
    if not folder.exists() or not qfile.exists():
        print(f"[略過] {name}：找不到 {folder.name} 或 {qfile.name}")
        continue
    docs = rc.load_collection(ROOT, cfg["collection"])
    qs = rc.load_jsonl(qfile)
    ids = {d["doc_id"] for d in docs}
    bad = [q["qid"] for q in qs if not set(rc.gold_ids(q)) <= ids]
    assert not bad, f"{name}: {len(bad)} 題的正確段落不在知識庫裡"
    data[name] = {"docs": docs, "doc_by_id": {d["doc_id"]: d for d in docs}, "questions": qs}
    n_gold = pd.Series([len(rc.gold_ids(q)) for q in qs]).value_counts().to_dict()
    print(f"{name}: {len(docs)} 段, {len(qs)} 題, 每題正確段落數 {n_gold}")

q_noans = rc.load_jsonl(ROOT / "eval" / NOANSWER["questions"])
print(f"無答案題: {len(q_noans)} 題（拿去問 {NOANSWER['ask_collection']}）")

SPLIT = holdout　RETRIEVER = hybrid　N_E2E = 100　題目檔：['squad_holdout100.jsonl', 'hotpot_holdout.jsonl', 'squad_b_noanswer_holdout50.jsonl']
squad: 534 段, 100 題, 每題正確段落數 {1: 100}
hotpot: 3954 段, 100 題, 每題正確段落數 {2: 100}
無答案題: 50 題（拿去問 squad_v11_dev500）


## 2. 索引與檢索器

`make_retriever()` 是唯一會依 `RETRIEVER` 改變的地方。

In [14]:
embedder = rc.Embedder(EMBED_MODEL)


def make_retriever(name: str, docs: list[dict], collection: str):
    if RETRIEVER in ("dense", "hybrid"):
        idx = rc.build_or_load_index(ROOT, docs, collection, embedder)
        dense = rc.DenseRetriever(idx, docs, embedder)
    if RETRIEVER in ("bm25", "hybrid"):
        bm25 = rc.BM25Retriever(docs, collection)
    ret = {"dense": lambda: dense, "bm25": lambda: bm25,
           "hybrid": lambda: rc.HybridRetriever(dense, bm25, rrf_k=RRF_K)}[RETRIEVER]()
    print(f"  {name}: {ret.index_key}")
    return ret


for name, d in data.items():
    d["retriever"] = make_retriever(name, d["docs"], DATASETS[name]["collection"])

  squad: squad_v11_dev500__BAAI-bge-small-en-v1-5__153e36d1e3b4+bm25-title+rrf60
  hotpot: hotpot_dev400__BAAI-bge-small-en-v1-5__7a77db1d36b0+bm25-title+rrf60


## 3. 檢索評估（全部題目，不呼叫 LLM）

In [15]:
ret_rows = []
for name, d in data.items():
    for q in d["questions"]:
        golds = rc.gold_ids(q)
        ranks = d["retriever"].ranks_of(q["question"], golds)
        ret_rows.append({"dataset": name, "qid": q["qid"], "type": q.get("type") or "single",
                         "question": q["question"], "gold": golds, "ranks": ranks,
                         "worst_rank": max(ranks), "best_rank": min(ranks)})
ret = pd.DataFrame(ret_rows)


def retrieval_table(df):
    out = {}
    n_gold = df.gold.map(len)
    for k in RETRIEVAL_KS:
        valid = df[n_gold <= k]                                # k 小於正確段落數時不可能全找到，不計
        out[f"hit@{k}"] = (valid.worst_rank <= k).mean() if len(valid) else np.nan   # 全部正確段落都在 Top-k
    for k in (3, 5):
        out[f"any@{k}"] = (df.best_rank <= k).mean()         # 至少一個在 Top-k
    out["MRR（第一個正確段落）"] = (1 / df.best_rank).mean()
    out["題數"] = len(df)
    return pd.Series(out)


groups = {f"{n}": g for n, g in ret.groupby("dataset")}
groups.update({f"hotpot / {t}": g for t, g in ret[ret.dataset == "hotpot"].groupby("type")})
ret_summary = pd.DataFrame({k: retrieval_table(v) for k, v in groups.items()}).round(3)
ret_summary

,hotpot,squad,hotpot / bridge,hotpot / comparison
hit@1,NaN,0.780,NaN,NaN
hit@2,0.430,0.910,0.461,0.333
hit@3,0.580,0.910,0.553,0.667
hit@5,0.700,0.910,0.658,0.833
hit@10,0.900,0.960,0.882,0.958
any@3,0.960,0.910,0.974,0.917
any@5,0.990,0.910,0.987,1.000
MRR（第一個正確段落）,0.925,0.854,0.943,0.868
題數,100.000,100.000,76.000,24.000


## 4. 完整問答（呼叫 LLM）

每題跑 `rag`（Top-k 檢索）和 `oracle`（直接給全部正確段落）；另外 50 題無答案題只跑 `rag`。

In [16]:
def run_e2e():
    gen = rc.OllamaGenerator(GEN_MODEL, OLLAMA_URL, NUM_CTX, GEN_THINK)
    gen.check()
    jobs = []
    for name, d in data.items():
        for q in d["questions"][:N_E2E]:
            jobs.append((name, q, "rag"))
            jobs.append((name, q, "oracle"))
            if RUN_NO_RAG:
                jobs.append((name, q, "no_rag"))
    noans_ds = next((n for n, c in DATASETS.items() if c["collection"] == NOANSWER["ask_collection"]), None)
    if noans_ds in data:
        jobs += [("noanswer", q, "rag") for q in q_noans]
    recs, t0 = [], time.time()
    for i, (name, q, mode) in enumerate(jobs, 1):
        d = data[noans_ds] if name == "noanswer" else data[name]
        gold_docs = [d["doc_by_id"][g] for g in rc.gold_ids(q)] if mode == "oracle" else None
        r = rc.ask(q, mode, gen, d["retriever"], k=TOP_K, char_budget=CONTEXT_CHAR_BUDGET, gold_docs=gold_docs)
        r.update(dataset=name, type=q.get("type") or "single", retriever=RETRIEVER)
        recs.append(r)
        if i % 50 == 0 or i == len(jobs):
            print(f"{i}/{len(jobs)}，{time.time() - t0:.0f}s")
    return recs


if LOAD_RUN is None:
    run_id = datetime.now().strftime("%Y%m%d_%H%M%S") + f"_bench_{RETRIEVER}_{SPLIT}"
    run_dir = ROOT / "results" / run_id
    run_dir.mkdir(parents=True, exist_ok=True)
    (run_dir / "config.json").write_text(json.dumps({
        "run_id": run_id, "split": SPLIT, "retriever": RETRIEVER, "rrf_k": RRF_K, "datasets": DATASETS, "noanswer": NOANSWER,
        "embed_model": EMBED_MODEL, "gen_model": GEN_MODEL, "gen_think": GEN_THINK, "top_k": TOP_K,
        "num_ctx": NUM_CTX, "context_char_budget": CONTEXT_CHAR_BUDGET, "n_e2e": N_E2E if RUN_E2E else 0,
        "run_no_rag": RUN_NO_RAG, "index_keys": {n: d["retriever"].index_key for n, d in data.items()},
        "system_rag": rc.SYSTEM_RAG}, indent=2, ensure_ascii=False), encoding="utf-8")
    ret.to_json(run_dir / "retrieval.jsonl", orient="records", lines=True, force_ascii=False)
    records = run_e2e() if RUN_E2E else []
    if records:
        with (run_dir / "records.jsonl").open("w", encoding="utf-8") as f:
            for r in records:
                f.write(json.dumps(r, ensure_ascii=False) + "\n")
    print("已存到", run_dir, "" if RUN_E2E else "（RUN_E2E = False：只有檢索結果）")
else:
    run_id, run_dir = LOAD_RUN, ROOT / "results" / LOAD_RUN
    rec_file = run_dir / "records.jsonl"
    records = rc.load_jsonl(rec_file) if rec_file.exists() else []
    print(f"讀取 {run_id}：{len(records)} 筆")

50/450，120s
100/450，241s
150/450，362s
200/450，481s
250/450，602s
300/450，723s
350/450，844s
400/450，966s
450/450，1083s
已存到 <專案根目錄>\results\20261026_191449_bench_hybrid_holdout 


## 5. 結果（完整問答）

`答對` = 回答包含標準答案，或 `f1_qstrip ≥ 0.5`（兩個自動指標任一成立）。仍然只是粗篩，失敗案例要人工看。

自動分類規則（只看 `rag` 模式）：
- `ok`：答對
- `retrieval_failure`：正確段落沒有全部進到 Context（HotpotQA 再細分成「一個都沒找到」和「只找到一個」）
- `generation_failure`：正確段落都在，oracle 也答錯
- `context_noise`：正確段落都在，oracle 答對、rag 答錯（被其他段落干擾）

`RUN_E2E = False` 時這一節會略過。

In [17]:
def add_flags(records):
    df = pd.DataFrame(records)
    if df.empty:
        return df
    df["correct"] = df.contains_gold.fillna(False).astype(bool) | (df.f1_qstrip.fillna(0) >= CORRECT_F1)
    df["abstained"] = df.raw_output.map(lambda t: rc.is_abstain(rc.parse_output(t)[0]))   # 用最新規則重算
    return df


def e2e_table(g):
    rag, ora, nor = g[g["mode"] == "rag"], g[g["mode"] == "oracle"], g[g["mode"] == "no_rag"]
    return pd.Series({
        "題數": len(rag),
        "答對（RAG）": rag.correct.mean(),
        "答對（oracle）": ora.correct.mean() if len(ora) else np.nan,
        "答對（No-RAG）": nor.correct.mean() if len(nor) else np.nan,
        "正確段落全在 Context": rag.gold_in_context.mean(),
        "拒答（全部）": rag.abstained.mean(),
        "拒答，但正確段落都在": (rag.abstained & rag.gold_in_context).mean(),
        "有寫 Sources": rag.has_sources.mean(),
        "平均秒數": rag.latency_s.mean(),
    })


def e2e_summary_of(df):
    ans = df[df.dataset != "noanswer"]
    groups = {n: g for n, g in ans.groupby("dataset")}
    groups.update({f"hotpot / {t}": g for t, g in ans[ans.dataset == "hotpot"].groupby("type")})
    return pd.DataFrame({k: e2e_table(v) for k, v in groups.items()}).round(3)


df = add_flags(records)
has_ans = not df.empty and (df.dataset != "noanswer").any()
if df.empty:
    print("沒有完整問答紀錄（RUN_E2E = False），略過第 5 節")
elif not has_ans:
    print("這次只跑無答案題（N_E2E = 0）")
else:
    display(e2e_summary_of(df))

,hotpot,squad,hotpot / bridge,hotpot / comparison
題數,100.00,100.000,76.000,24.000
答對（RAG）,0.51,0.850,0.579,0.292
答對（oracle）,0.72,0.950,0.789,0.500
答對（No-RAG）,NaN,NaN,NaN,NaN
正確段落全在 Context,0.58,0.910,0.553,0.667
拒答（全部）,0.35,0.070,0.289,0.542
拒答，但正確段落都在,0.10,0.040,0.053,0.250
有寫 Sources,0.91,0.970,0.947,0.792
平均秒數,2.42,2.446,2.425,2.405


In [18]:
if not df.empty:
    noans = df[df.dataset == "noanswer"]
    if len(noans):
        print(f"無答案題：拒答 {int(noans.abstained.sum())}/{len(noans)}，亂答 {int((~noans.abstained).sum())}")
        display(noans[~noans.abstained][["question", "answer", "cited_ids"]])

無答案題：拒答 49/50，亂答 1


,question,answer,cited_ids
447,What things did the network concentrate on,"behavioral and demographic data, sponsor selection, programming for certain audiences",[American_Broadcasting_Company__p043]


In [19]:
def label_runs(df):
    ans = df[df.dataset != "noanswer"]
    rag = ans[ans["mode"] == "rag"].set_index(["dataset", "qid"])
    ora = ans[ans["mode"] == "oracle"].set_index(["dataset", "qid"])

    def classify(key):
        r = rag.loc[key]
        if r.correct:
            return "ok"
        if not r.gold_in_context:
            if len(r.gold_doc_ids) > 1:
                return "retrieval_failure（只找到一個）" if r.gold_found else "retrieval_failure（都沒找到）"
            return "retrieval_failure"
        return "context_noise" if ora.loc[key].correct else "generation_failure"

    rag = rag.copy()
    rag["auto_label"] = [classify(k) for k in rag.index]
    return rag, ora


if has_ans:
    rag, ora = label_runs(df)
    display(rag.reset_index().groupby(["dataset", "auto_label"]).size().unstack(0).fillna(0).astype(int))

dataset,hotpot,squad
auto_label,,
context_noise,4,4
generation_failure,13,3
ok,51,85
retrieval_failure,0,8
retrieval_failure（只找到一個）,29,0
retrieval_failure（都沒找到）,3,0


### 匯出失敗案例

所有 `auto_label` 不是 `ok` 的題目，連同正確段落和拿到的段落 ID，存成 CSV 方便之後人工核對或做檢視頁。

In [20]:
if has_ans:
    fail = rag[rag.auto_label != "ok"].reset_index()
    fail["oracle_answer"] = [ora.loc[(r.dataset, r.qid)].answer for r in fail.itertuples()]
    cols = ["dataset", "type", "qid", "question", "gold_answers", "gold_doc_ids", "context_ids",
            "answer", "oracle_answer", "cited_ids", "auto_label"]
    out = ROOT / "reports" / f"failures_{run_id}.csv"
    out.parent.mkdir(exist_ok=True)
    fail[cols].to_csv(out, index=False, encoding="utf-8-sig")
    print(f"{len(fail)} 題，已匯出 {out}")

64 題，已匯出 <專案根目錄>\reports\failures_20261026_191449_bench_hybrid_holdout.csv


## 6. 怎麼讀這些數字

- **第 3 節是檢索方法比較的主角。** 不受 LLM 隨機性影響；Stage 3 主要比較 `hit@k` 和 MRR。
- **HotpotQA 的 `hit@3` 會比 SQuAD 低很多**，因為兩個段落都要找到。看 `any@3` 和 `hit@3` 的差距：差很多代表常常只找到一半。
- **`答對（oracle）` 是生成端的上限。** 如果 HotpotQA 的 oracle 也不高，代表 4B 的多步推理本身就有困難，再好的檢索也救不了這部分。
- **`答對（RAG）` 和 `答對（oracle）` 之間的差距**，就是檢索改善最多能帶來的空間。
- **`答對（No-RAG）`**（`RUN_NO_RAG = True` 才有）：模型不看文件就答對的比例。這個數字高，代表模型可能在預訓練時看過這些題目。
- **無答案題**是 Stage 3 的副作用檢查：換成混合檢索後，亂答不應該變多。

## 7. 和基準 run 並排比較

把第 0 節的 `BASELINE_RUN` 設成基準那次的 run_id（例如 dense 的 `20261026_001846_bench_dense`）。

- **檢索**：兩次的 hit@k、MRR 並排，以及「這次救回幾題、弄壞幾題」。檢索結果是固定的，差一題就是真的差一題。
- **完整問答**：兩次的答對率並排。LLM 每次結果略有不同，**差 2～3 題以內不要下結論**；看「救回／弄壞」的題目內容比看平均更可靠。

In [21]:
base_dir = ROOT / "results" / BASELINE_RUN if BASELINE_RUN else None
if base_dir is None or not (base_dir / "retrieval.jsonl").exists():
    print("沒有設定 BASELINE_RUN，或找不到它的 retrieval.jsonl，略過第 7 節")
else:
    base_cfg = json.loads((base_dir / "config.json").read_text(encoding="utf-8"))
    base_ret = pd.read_json(base_dir / "retrieval.jsonl", lines=True)
    print(f"基準：{BASELINE_RUN}（{base_cfg['retriever']}）  vs  這次：{run_id}（{RETRIEVER}）")
    if base_cfg.get("split", "dev") != SPLIT:
        print(f"⚠ 基準是 {base_cfg.get('split', 'dev')}、這次是 {SPLIT}：題目不同，下面的比較只算兩邊都有的題目")

    rows = {}
    for name in sorted(set(ret.dataset)):
        rows[(name, base_cfg["retriever"])] = retrieval_table(base_ret[base_ret.dataset == name])
        rows[(name, RETRIEVER)] = retrieval_table(ret[ret.dataset == name])
    display(pd.DataFrame(rows).round(3))

    m = ret.merge(base_ret[["dataset", "qid", "worst_rank"]], on=["dataset", "qid"], suffixes=("", "_base"))
    for name, g in m.groupby("dataset"):
        fixed = g[(g.worst_rank <= TOP_K) & (g.worst_rank_base > TOP_K)]
        broken = g[(g.worst_rank > TOP_K) & (g.worst_rank_base <= TOP_K)]
        print(f"{name}（k={TOP_K}）：救回 {len(fixed)} 題，弄壞 {len(broken)} 題")
        if len(broken):
            display(broken[["question", "worst_rank_base", "worst_rank"]].head(10))

基準：20261026_185546_bench_dense_holdout（dense）  vs  這次：20261026_191449_bench_hybrid_holdout（hybrid）


hotpot             squad         
                dense   hybrid    dense   hybrid
hit@1             NaN      NaN    0.700    0.780
hit@2           0.620    0.430    0.810    0.910
hit@3           0.790    0.580    0.890    0.910
hit@5           0.870    0.700    0.920    0.910
hit@10          0.960    0.900    0.960    0.960
any@3           0.970    0.960    0.890    0.910
any@5           1.000    0.990    0.920    0.910
MRR（第一個正確段落）    0.933    0.925    0.795    0.854
題數            100.000  100.000  100.000  100.000

hotpot（k=3）：救回 1 題，弄壞 22 題


,question,worst_rank_base,worst_rank
100,Mount Stimson is the second highest peak in national park that encompasses how many ac...,2,6
101,"""Quando m'en vo' "" is a soprano from an Opera composed by who ?",2,4
112,Are Chrysalis and Look both women's magazines?,2,4
114,"""It's Me That You Need"" is a song with lyrics written by Bernie Taupin and which artis...",3,7
118,What conference does Ralphie the Buffalo's football team play in?,3,27
121,David Huntsinger has worked with this gospel singer born in the month of July?,3,7
122,When was the album released for which Lady Gaga used for material for her Super Bowl L...,2,5
123,Red Mountain is a film that centers on an attempt by a group of what group of Civil Wa...,3,7
126,What county in Minnesota holds a 202 lb slab of greywacke covered in runes?,2,4
127,The 2010–11 Old Dominion Monarchs basketball team was led by the coach of what nationa...,2,5


squad（k=3）：救回 6 題，弄壞 4 題


,question,worst_rank_base,worst_rank
13,Which areas of Northern Europe practiced those religions?,3,12
14,Where did these uprisings take place?,2,20
55,What is the present-day location of this church?,3,6
71,How do plants get chloroplasts?,3,11


In [22]:
if base_dir is not None and (base_dir / "records.jsonl").exists() and not df.empty:
    base_df = add_flags(rc.load_jsonl(base_dir / "records.jsonl"))
    if has_ans and (base_df.dataset != "noanswer").any():
        cmp = pd.concat({base_cfg["retriever"]: e2e_summary_of(base_df), RETRIEVER: e2e_summary_of(df)}, axis=1)
        display(cmp.swaplevel(axis=1).sort_index(axis=1, level=0))

    for label, d in ((base_cfg["retriever"], base_df), (RETRIEVER, df)):
        na = d[d.dataset == "noanswer"]
        if len(na):
            print(f"無答案題（{label}）：拒答 {int(na.abstained.sum())}/{len(na)}")
    na_b = base_df[base_df.dataset == "noanswer"].set_index("qid")
    na_n = df[df.dataset == "noanswer"].set_index("qid")
    common = na_b.index.intersection(na_n.index)
    new_halluc = [q for q in common if na_b.loc[q].abstained and not na_n.loc[q].abstained]
    if len(common):
        print(f"無答案題（兩次都有的 {len(common)} 題）：這次新出現的亂答 {len(new_halluc)} 題")
        if new_halluc:
            display(na_n.loc[new_halluc][["question", "answer", "cited_ids"]])

    pick = lambda d: d[(d["mode"] == "rag") & (d.dataset != "noanswer")].set_index(["dataset", "qid"]).correct
    a, b = pick(base_df), pick(df)
    both = pd.concat({"base": a, "now": b}, axis=1).dropna()
    for name, g in both.groupby(level=0):
        print(f"{name}：答對 救回 {int((~g.base.astype(bool) & g.now.astype(bool)).sum())} 題，"
              f"弄壞 {int((g.base.astype(bool) & ~g.now.astype(bool)).sum())} 題")

hotpot         hotpot / bridge         hotpot / comparison  \
                  dense  hybrid           dense  hybrid               dense   
題數              100.000  100.00          76.000  76.000              24.000   
答對（RAG）           0.620    0.51           0.671   0.579               0.458   
答對（oracle）        0.730    0.72           0.789   0.789               0.542   
答對（No-RAG）          NaN     NaN             NaN     NaN                 NaN   
正確段落全在 Context    0.790    0.58           0.776   0.553               0.833   
拒答（全部）            0.260    0.35           0.237   0.289               0.333   
拒答，但正確段落都在        0.120    0.10           0.092   0.053               0.208   
有寫 Sources        0.920    0.91           0.947   0.947               0.833   
平均秒數              2.434    2.42           2.431   2.425               2.441   

                          squad           
                hybrid    dense   hybrid  
題數              24.000  100.000  100.000  
答對（RAG）          0.292    0.830    0.850  
答對（oracle）       0.500    0.950    0.950  
答對（No-RAG）         NaN      NaN      NaN  
正確段落全在 Context   0.667    0.890    0.910  
拒答（全部）           0.542    0.100    0.070  
拒答，但正確段落都在       0.250    0.020    0.040  
有寫 Sources       0.792    0.950    0.970  
平均秒數             2.405    2.477    2.446

無答案題（dense）：拒答 49/50
無答案題（hybrid）：拒答 49/50
無答案題（兩次都有的 50 題）：這次新出現的亂答 0 題
hotpot：答對 救回 1 題，弄壞 12 題
squad：答對 救回 6 題，弄壞 4 題
